# Day 1 — Exploring the Grounding Corpus

Goal: load all three scraped sources (`ncfe_faqs.json`, `sebi_articles.json`, `wikipedia_articles.json`), understand their shape before deciding chunking strategy (Day 2). No embedding/FAISS work here yet — just looking at what we have.

In [ ]:
import json
from collections import Counter

ncfe = json.load(open("../data/raw/ncfe_faqs.json"))
sebi = json.load(open("../data/raw/sebi_articles.json"))
wiki = json.load(open("../data/raw/wikipedia_articles.json"))

print(f"NCFE: {len(ncfe)} Q&A pairs")
print(f"SEBI: {len(sebi)} articles")
print(f"Wikipedia: {len(wiki)} articles")
print(f"Total corpus entries: {len(ncfe) + len(sebi) + len(wiki)}")


NCFE: 67 Q&A pairs
SEBI: 6 articles
Wikipedia: 9 articles
Total corpus entries: 82


Three sources, three different shapes:
- **NCFE**: 67 short Q&A pairs (natural chunk = one pair)
- **SEBI**: 6 short-to-medium articles (mostly landing-page-style overviews, one substantial one — "Caution to Investor", ~5.4k chars, covers Ponzi schemes / unregistered advisers — directly relevant to the "we want to learn trading to earn money" seed question)
- **Wikipedia**: 9 full encyclopedia articles, wildly different lengths (658 to 77k chars) — these will need real chunking (Day 2), not per-article-as-one-chunk

In [ ]:
print("NCFE categories:", Counter(d["category"] for d in ncfe))
ncfe_lens = [len(d["answer"]) for d in ncfe]
print(f"NCFE answer length - min: {min(ncfe_lens)}, max: {max(ncfe_lens)}, avg: {sum(ncfe_lens)//len(ncfe_lens)}")


NCFE categories: {'financial-planning': 10, 'banking': 8, 'loan-borrowing': 7, 'mutual-fund': 34, 'gold': 8}
NCFE answer length - min: 141, max: 13343, avg: 1985


In [ ]:
for d in ncfe[:2]:
    print("Q:", d["question"])
    print("A:", d["answer"][:250], "...")
    print("source:", d["source_url"])
    print("-" * 80)


Q: Financial Planning Overview
A: Financial Planning We all try to plan our lives as much as possible. We expect to finish studies in our early 20s, get a job, buy a house by the age of 27, get a car by 29 and so on. Our capacity to dream and aim is unlimited. This needs thorough pla ...
source: https://ncfe.org.in/faqs/financial-planning/
--------------------------------------------------------------------------------
Q: What are the broad areas in which financial planning can be undertaken?
A: Financial planning is not a simple task. You need to take into account multiple factors about your life – past, present and future – in order to form a feasible financial plan. Remember, for a plan to be effective, it has to be well-thought, comprehe ...
source: https://ncfe.org.in/faqs/financial-planning/
--------------------------------------------------------------------------------


## SEBI articles — all 6 (secondary source, no explicit reproduction prohibition, always cite)

In [ ]:
for d in sebi:
    print("Title:", d["title"], f"({len(d['text'])} chars)")
    print("Text:", d["text"][:200], "...")
    print("-" * 80)


Title: Personal Finance & Investments : Money matters: Lets Understand (1247 chars)
Text: What You Will Learn This section provides you insights on personal finance that will help you to take control of your money and secure a better future for yourself and your family. This will cover top ...
--------------------------------------------------------------------------------
Title: Personal Finance & Investment : Investments : Lets Understand (1151 chars)
Text: What You Will Learn This section provides you an essential guide on investment and investment assets class that will help you grow your money and secure a better future for yourself and your family. T ...
--------------------------------------------------------------------------------
Title: Personal Finance & Investment : Securities Market Investments : Let's Learn (1098 chars)
Text: What you will Learn This section provides you an essential guide on another avenue for investment i.e. securities market that will help you grow you

## Wikipedia articles — all 9 (CC BY-SA, supplementary source)

In [ ]:
for d in wiki:
    print("Title:", d["title"], f"({len(d['text'])} chars)")
    print("Text:", d["text"][:200].replace(chr(10), " "), "...")
    print("-" * 80)


Title: Systematic investment plan (4647 chars)
Text: A systematic investment plan (SIP) is an investment vehicle offered by many mutual funds to investors, allowing them to invest small amounts periodically instead of lump sums. The frequency of investm ...
--------------------------------------------------------------------------------
Title: Public Provident Fund (India) (10526 chars)
Text: The Public Provident Fund (PPF) is a voluntary savings-tax-reduction social security instrument in India, introduced by the National Savings Institute of the Ministry of Finance in 1968. The scheme's  ...
--------------------------------------------------------------------------------
Title: National Pension System (14654 chars)
Text: The National Pension System (NPS) is a defined-contribution pension system in India regulated by the Pension Fund Regulatory and Development Authority (PFRDA) which is under the jurisdiction of the Mi ...
--------------------------------------------------------------

## Observations for Day 2 (chunking strategy)

- **NCFE**: per-Q&A-pair chunking (already decided) — every entry is already a natural, self-contained unit.
- **SEBI**: per-article chunking works for 5 of 6 (all under ~1.3k chars); the "Caution to Investor" article (5.4k chars) covers multiple scam types in one page and may benefit from splitting by its internal topic (Ponzi schemes / unregistered advisers / etc.) rather than embedding it as one chunk.
- **Wikipedia**: real chunking required. `Credit card` (77k chars) and `Mutual fund` (34k chars) cannot be single chunks — need fixed-size-with-overlap or per-heading-section chunking (the extracts already include `== Heading ==` markers from the MediaWiki API, which makes per-section splitting straightforward). Short articles like `Emergency fund` (658 chars) are fine as single chunks.
- **Net decision for `ingest.py` (Day 2):** not a single uniform strategy — per-Q&A-pair for NCFE, per-article for short SEBI/Wikipedia entries, and size-based splitting (by heading, falling back to fixed-size-with-overlap) for anything over roughly 1.5k-2k chars.